# FQL Succession P2 — **C-1: FQL `distill_alpha_bc` rescue sweep** (clean E-uni)

**Purpose**: the *fairness rematch* (direction **C**) from `docs/fql_succession_p2_mechanism_diagnostic.md` **§9.10** — give FQL its own BC-anchor tuning, **one honest chance to overturn Q1c's verdict** before we finalize a negative/mechanism result.

## Background — why FQL needs a rematch

Q1c (§9.7) showed **ReBRAC β1=1.0 dominates FQL on every cell** (worst-case-over-noise **0.910 > FQL 0.855**), killing the "FQL wins" claim. **But** Gate B tuned ReBRAC's β1 while **freezing FQL's `distill_alpha_bc=1.0`** and never sweeping it — an **asymmetric-tuning gap** (§9.7 fairness caveat). C closes that gap.

FQL's knob is the BC-distillation weight in

> `actor_loss = −λ·Q̄ + distill_alpha_bc · ‖a_student − a_teacher‖²`  (fql.py:472–474)

the direct analog of ReBRAC's β1 — except it anchors to the flow-**denoised** teacher action rather than raw data. CLI `--distill-alpha-bc`, default **1.0**.

## The bar FQL must clear

Worst-case-over-noise must exceed **0.910** (ReBRAC β1=1.0). FQL's worst-case is its **clean** cell (0.855), so **clean is the binding axis** to improve. This notebook sweeps alpha on **clean E-uni only** (staged — the noisy axis is **Phase C-2**, a follow-up notebook built *only if* a candidate emerges).

Direction is genuinely uncertain: ReBRAC's lesson was *weaker* anchor helps (β1 4.0→1.0), but FQL anchors to a near-expert **denoised** teacher on clean data, where *stronger* anchoring could help → sweep **log-spaced both sides** of 1.0.

## Hypothesis (this notebook, clean axis)

- **RESCUE-PROMISING**: some α lifts FQL clean to **≥ 0.90** (within ~1 pp of the 0.910 bar) → carry that α into **Phase C-2** (noisy) to confirm worst-case.
- **RESCUE-WEAK**: best α clean in **[0.86, 0.90)** → improved but short; C-2 optional.
- **RESCUE-FAIL**: no α beats FQL's own α=1.0 clean (~0.855) → FQL cannot clear the binding axis → **fall back to B+A** (honest mechanism/negative result).

## Design — 8 FQL train+eval runs

| group | `--distill-alpha-bc` | seeds | dataset | tree | runs |
|---|---|---|---|---|---|
| SWEEP | **0.3, 3.0, 10.0** | [42, 0] | E-uni **clean** (σ=0) | `e_uni_c1_fql_alpha_sweep` | 6 |
| TOP-UP | 1.0 (frozen) | **[1, 2]** new | E-uni **clean** (σ=0) | extends `e_uni` → **n=4** | 2 |

TOP-UP matches the original E-uni FQL config **exactly** (only seed differs) so it is a valid extension of that baseline (§9.9: FQL-clean is the sole under-determined cell, 0.80↔0.91 at n=2). Wallclock ≈ 30–45 min/run on L4 → ~4–6 h, 1–2 sessions.

## Out of scope

- **Noisy axis** (M-uni-noise) — Phase C-2 follow-up, only if C-1 finds a candidate.
- ReBRAC re-run — the β1=1.0 clean bar (0.910) is already in `e_uni_q1c_actor_pen1_clean/test`.
- No edits to Gate-B-frozen `auv_nav/{fql,rebrac}.py` (only the CLI value changes).

## 1. Environment sanity

确认 cwd 是 repo root + GPU 可用 + `auv_nav` import 通过。

In [ ]:
!nvidia-smi | head -15
!lscpu | head -10

Fri May 22 17:12:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
print(f'torch={torch.__version__}  CUDA available={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device={torch.cuda.get_device_name(0)}')

torch=2.10.0+cu128  CUDA available=True
device=NVIDIA L4


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [ ]:
%cd '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'

/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


In [ ]:
import os, sys, subprocess
from pathlib import Path

if Path.cwd().name != 'rl_v2':
    candidates = [
        Path('/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'),
        Path('/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2'),
    ]
    for c in candidates:
        if c.exists():
            os.chdir(c); break
    else:
        raise RuntimeError('repo root not found')

print('cwd =', Path.cwd())
import torch
print('torch =', torch.__version__, '  cuda =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  device =', torch.cuda.get_device_name(0))
import auv_nav
print('auv_nav OK')

cwd = /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5
torch = 2.10.0+cu128   cuda = True
  device = NVIDIA L4
auv_nav OK


## 2. Config

共享参数与原 E-uni run **完全一致**(probe s0 / h4 / target 1.5 / cross_stream / arrival_v2 / 200k steps / batch 256 / eval-every 10k / eval-episodes 100)。FQL flags 也冻结,**唯一变动是 `--distill-alpha-bc`**。

- SWEEP:α ∈ {0.3, 3.0, 10.0} × seeds [42, 0] → `e_uni_c1_fql_alpha_sweep` 树。
- TOP-UP:α=1.0 × seeds **[1, 2]**(新)→ **就地扩** 现有 `e_uni` cell 到 n=4。

In [ ]:
from pathlib import Path

DATASET  = 'offline_data/privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000'

FLOW     = 'wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
MANIFEST = 'benchmarks/single_u10_cross_tgt15_ep100.json'

PROBE_LAYOUT   = 's0'
HISTORY_LENGTH = 4
TARGET_SPEED   = 1.5
TASK_GEOMETRY  = 'cross_stream'
OBJECTIVE      = 'arrival_v2'

# Matches the original E-uni FQL runs EXACTLY (so the α=1.0 top-up extends that baseline).
TOTAL_STEPS    = 200_000
BATCH_SIZE     = 256
EVAL_EVERY     = 10_000
EVAL_EPISODES  = 100   # in-training monitoring; test eval (§5) is the comparison metric
TEST_EPISODES  = 100
EVAL_NUM_WORKERS = 6

# FQL Gate-B-frozen flags; ONLY --distill-alpha-bc varies in this notebook.
def fql_flags(alpha):
    return (
        f'--flow-steps 10 --distill-alpha-bc {alpha} '
        f'--teacher-lr 3e-4 --flow-time-embed-dim 32'
    )

def atag(alpha):
    # 0.3->'0p3', 1.0->'1p0', 3.0->'3p0', 10.0->'10p0'
    return str(alpha).replace('.', 'p')

ALPHAS_SWEEP   = [0.3, 3.0, 10.0]   # new alphas, clean E-uni
ALPHA_BASELINE = 1.0                # frozen value, for the seed top-up
TRAIN_SEEDS    = [42, 0]            # baseline seeds (existing for α=1.0; reused for sweep)
TOPUP_SEEDS    = [1, 2]             # NEW seeds: bump α=1.0 clean to n=4 (§9.9)

# Two output trees:
SWEEP_CKPT = Path('checkpoints/fql_succession/p2/e_uni_c1_fql_alpha_sweep')
SWEEP_RES  = Path('results/fql_succession/p2/e_uni_c1_fql_alpha_sweep')
EUNI_CKPT  = Path('checkpoints/fql_succession/p2/e_uni')   # existing baseline cell
EUNI_RES   = Path('results/fql_succession/p2/e_uni')

MIRROR_FILES = ('train_log.jsonl', 'eval_log.csv', 'trainer_state.json', 'train_config.txt')

def make_job(alpha, seed, ckpt_root, res_root, name):
    return {
        'alpha': alpha, 'seed': seed, 'name': name,
        'ckpt': ckpt_root / name,
        'test_json': res_root / 'test' / f'{name}.json',
        'mirror': res_root / 'training_curves' / name,
    }

JOBS = []
# SWEEP: new alphas at baseline seeds → sweep tree, name fql_a<tag>_seed<S>
for alpha in ALPHAS_SWEEP:
    for seed in TRAIN_SEEDS:
        JOBS.append(make_job(alpha, seed, SWEEP_CKPT, SWEEP_RES, f'fql_a{atag(alpha)}_seed{seed}'))
# TOP-UP: α=1.0 at NEW seeds → extend existing e_uni cell, name fql_seed<S> (matches baseline)
for seed in TOPUP_SEEDS:
    JOBS.append(make_job(ALPHA_BASELINE, seed, EUNI_CKPT, EUNI_RES, f'fql_seed{seed}'))

print(f'{len(JOBS)} jobs:')
for j in JOBS:
    print(f"  α={j['alpha']:<5} seed={j['seed']:<3} → {j['ckpt']}")

8 jobs:
  α=0.3   seed=42  → checkpoints/fql_succession/p2/e_uni_c1_fql_alpha_sweep/fql_a0p3_seed42
  α=0.3   seed=0   → checkpoints/fql_succession/p2/e_uni_c1_fql_alpha_sweep/fql_a0p3_seed0
  α=3.0   seed=42  → checkpoints/fql_succession/p2/e_uni_c1_fql_alpha_sweep/fql_a3p0_seed42
  α=3.0   seed=0   → checkpoints/fql_succession/p2/e_uni_c1_fql_alpha_sweep/fql_a3p0_seed0
  α=10.0  seed=42  → checkpoints/fql_succession/p2/e_uni_c1_fql_alpha_sweep/fql_a10p0_seed42
  α=10.0  seed=0   → checkpoints/fql_succession/p2/e_uni_c1_fql_alpha_sweep/fql_a10p0_seed0
  α=1.0   seed=1   → checkpoints/fql_succession/p2/e_uni/fql_seed1
  α=1.0   seed=2   → checkpoints/fql_succession/p2/e_uni/fql_seed2


## 3. Pre-flight

确认:
1. Dataset dir + `transitions.npz` + `metadata.json` exist
2. metadata 是 **CLEAN**(σ=0.0)+ unimodal + ~86685 transitions(与 baseline 同源)
3. manifest + flow exist
4. 打印 FQL α=1.0 baseline(seeds 42,0)+ ReBRAC β1=1.0 clean bar(0.910)供对照 — 不强制

In [ ]:
import json
from pathlib import Path
from statistics import mean

ds = Path(DATASET)
assert ds.is_dir(), f'dataset dir missing: {ds}'
assert (ds / 'transitions.npz').is_file(), f'transitions.npz missing under {ds}'
meta_path = ds / 'metadata.json'
assert meta_path.is_file(), f'metadata.json missing under {ds}'
meta = json.loads(meta_path.read_text())
print('policy_mixture   =', meta.get('policy_mixture'))
print('action_noise_std =', meta.get('action_noise_std'))
print('num_transitions  =', meta.get('num_transitions'))
print('source_dataset   =', ds.name)

# MUST be clean (σ=0) — the rescue sweep is on the CLEAN binding axis.
noise = float(meta.get('action_noise_std', float('nan')))
assert abs(noise) < 1e-6, (
    f'expected action_noise_std==0.0 (clean) but got {noise} '
    '— C-1 sweeps FQL on CLEAN E-uni (its worst-case axis).'
)
exp_n_tx_approx = 86685
n_tx = int(meta.get('num_transitions', 0))
assert abs(n_tx - exp_n_tx_approx) < 5000, f'tx count {n_tx} far from expected {exp_n_tx_approx}'

assert Path(FLOW).is_file(), f'flow missing: {FLOW}'
assert Path(MANIFEST).is_file(), f'manifest missing: {MANIFEST}'

EUNI_TEST = Path('results/fql_succession/p2/e_uni/test')
Q1C_DIR   = Path('results/fql_succession/p2/e_uni_q1c_actor_pen1_clean/test')
def _sr(p): return float(json.loads(p.read_text())['eval_success_rate']) if p.exists() else None
print()
for s in TRAIN_SEEDS:
    print(f'FQL  α=1.0 clean  seed={s}  SR =', _sr(EUNI_TEST / f'fql_seed{s}.json'))
rb = [_sr(Q1C_DIR / f'rebrac_seed{s}.json') for s in TRAIN_SEEDS]
rb = [v for v in rb if v is not None]
print('ReBRAC β1=1.0 clean BAR (mean) =', round(mean(rb), 3) if rb else None, '  <- FQL worst-case must exceed this')

print('\nPre-flight PASS — ready to train.')

policy_mixture   = [{'policy': 'privileged', 'weight': 1.0}]
action_noise_std = 0.0
num_transitions  = 86685
source_dataset   = privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000

FQL  α=1.0 clean  seed=42  SR = 0.8
FQL  α=1.0 clean  seed=0  SR = 0.91
ReBRAC β1=1.0 clean BAR (mean) = 0.91   <- FQL worst-case must exceed this

Pre-flight PASS — ready to train.


## 4. Train (8 runs: 6 sweep + 2 top-up)

FQL only,逐 job 跑;skip-resume on `agent_final.pt`(re-run 安全)。唯一变动是 `--distill-alpha-bc`(由 `fql_flags(alpha)` 注入)。

**`--skip-final-eval`**:在线 eval 跑 100 ep 监控,test eval 100 ep 走 §5。

In [ ]:
import time
from pathlib import Path

t_start_train = time.time()

for job in JOBS:
    alpha = job['alpha']; seed = job['seed']
    sd = job['ckpt']; sd_str = str(sd)
    FQLF = fql_flags(alpha)
    if (sd / 'agent_final.pt').exists():
        print(f'[skip-train] α={alpha} seed={seed} (agent_final.pt exists at {sd})')
        continue
    sd.mkdir(parents=True, exist_ok=True)
    print(f'\n{"=" * 72}')
    print(f'  train FQL α={alpha} seed={seed} → {sd}')
    print(f'{"=" * 72}')
    t0 = time.time()
    !python -m scripts.train_offline \
        --algo fql \
        --offline-data '{DATASET}/transitions.npz' \
        --flow '{FLOW}' \
        --manifest '{MANIFEST}' \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --target-speed {TARGET_SPEED} \
        --task-geometry {TASK_GEOMETRY} \
        --objective {OBJECTIVE} \
        --total-steps {TOTAL_STEPS} \
        --batch-size {BATCH_SIZE} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        {FQLF} \
        --skip-final-eval \
        --seed {seed} \
        --save-dir '{sd_str}' \
        --device cuda
    print(f'[train done] α={alpha} seed={seed} in {(time.time() - t0) / 60:.1f} min')

print(f'\n[all train] total = {(time.time() - t_start_train) / 60:.1f} min')


  train FQL α=0.3 seed=42 → checkpoints/fql_succession/p2/e_uni_c1_fql_alpha_sweep/fql_a0p3_seed42
[offline] algo=fql transitions=86685 obs_dim=48 action_dim=2 protocol=deployable tensor_replay=on eval_workers=1 sampling=uniform total_steps=200000
[train] step=1 q=0.110 critic=117.852 actor=-0.348 bc=1.060 lambda=6.046
[train] step=1000 q=6.000 critic=321.000 actor=-0.908 bc=0.233 lambda=0.163
[train] step=2000 q=11.725 critic=5.957 actor=-0.940 bc=0.173 lambda=0.085
[train] step=3000 q=19.347 critic=2.216 actor=-0.953 bc=0.139 lambda=0.051
[train] step=4000 q=24.111 critic=1.385 actor=-0.972 bc=0.067 lambda=0.041
[train] step=5000 q=31.756 critic=0.888 actor=-0.977 bc=0.045 lambda=0.031
[train] step=6000 q=36.923 critic=0.804 actor=-0.987 bc=0.031 lambda=0.027
[train] step=7000 q=36.185 critic=0.529 actor=-0.980 bc=0.045 lambda=0.027
[train] step=8000 q=40.901 critic=0.975 actor=-0.987 bc=0.022 lambda=0.024
[train] step=9000 q=39.212 critic=0.788 actor=-0.988 bc=0.015 lambda=0.025
[t

## 5. Test eval (100 ep / seed) + mirror small files

对每个 `agent_final.pt` 跑 `evaluate_offline` 100 ep(固定 manifest re-eval),写 `<test_json>`;同时 mirror 4 个 small files。CLI 与 sprint-0 §5 一致。

In [9]:
import time
import shutil
from pathlib import Path

t_start_eval = time.time()

for job in JOBS:
    alpha = job['alpha']; seed = job['seed']
    sd = job['ckpt']; sd_str = str(sd)
    out_path = job['test_json']; out_path_str = str(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists():
        print(f'[skip-eval] α={alpha} seed={seed}: {out_path}')
        continue
    if not (sd / 'agent_final.pt').exists():
        print(f'[warn] missing agent_final.pt at {sd} — training not done?')
        continue

    print(f'\n{"=" * 72}')
    print(f'  eval FQL α={alpha} seed={seed} → {out_path}')
    print(f'{"=" * 72}')
    t0 = time.time()
    !python -m scripts.evaluate_offline \
        --checkpoint '{sd_str}' \
        --manifest '{MANIFEST}' \
        --episodes {TEST_EPISODES} \
        --num-workers {EVAL_NUM_WORKERS} \
        --worker-device cpu \
        --device cuda \
        --output-json '{out_path_str}'
    print(f'[eval done] α={alpha} seed={seed} in {(time.time() - t0) / 60:.1f} min')

print(f'\n[all eval] total = {(time.time() - t_start_eval) / 60:.1f} min')

# Mirror 4 small files (always — outside the skip path)
for job in JOBS:
    sd = job['ckpt']; mdir = job['mirror']
    mdir.mkdir(parents=True, exist_ok=True)
    for fname in MIRROR_FILES:
        src = sd / fname
        if src.exists():
            shutil.copy2(src, mdir / fname)
    print(f"mirrored small files for α={job['alpha']} seed={job['seed']} → {mdir}")


  eval FQL α=0.3 seed=42 → results/fql_succession/p2/e_uni_c1_fql_alpha_sweep/test/fql_a0p3_seed42.json
reward_objective      : arrival_v2
energy_cost_gain      : 0.000000
safety_cost_gain      : 0.000000
episodes              : 100
success_rate          : 75.0%
avg_return            : 52.73 +/- 154.71
avg_safety_cost       : 4.214 +/- 4.336
avg_time_s            : 43.56 +/- 6.85
avg_time_s_success    : 41.10
avg_energy            : 34155.74 +/- 5973.66
avg_path_length_m     : 48.09 +/- 9.56
avg_progress_ratio    : 0.782 +/- 0.345
avg_path_efficiency   : 0.750 +/- 0.302
termination           : {'out_of_bounds': 25, 'goal': 75}
benchmark_manifest    : benchmarks/single_u10_cross_tgt15_ep100.json
[eval done] α=0.3 seed=42 in 1.2 min

  eval FQL α=0.3 seed=0 → results/fql_succession/p2/e_uni_c1_fql_alpha_sweep/test/fql_a0p3_seed0.json
reward_objective      : arrival_v2
energy_cost_gain      : 0.000000
safety_cost_gain      : 0.000000
episodes              : 100
success_rate          : 73

## 6. FQL clean success-rate vs `distill_alpha_bc`

把 4 个 α(0.3 / 1.0[n=4] / 3.0 / 10.0)的 **clean** test SR 排成一列,对照两条线:**FQL α=1.0 旧值 0.855** 和 **ReBRAC β1=1.0 clean bar 0.910**。

α=1.0 读 `e_uni/test`(seeds 42,0 旧 + 1,2 top-up = **n=4**);其余 α 读 sweep 树(seeds 42,0)。

In [10]:
import json
from pathlib import Path
from statistics import mean

EUNI_TEST  = Path('results/fql_succession/p2/e_uni/test')
Q1C_DIR    = Path('results/fql_succession/p2/e_uni_q1c_actor_pen1_clean/test')
SWEEP_TEST = SWEEP_RES / 'test'

def sr(p): return float(json.loads(p.read_text())['eval_success_rate']) if p.exists() else None

def alpha_cells(alpha):
    # returns (mean_or_None, n, dict{seed: sr})
    if alpha == ALPHA_BASELINE:
        seeds = TRAIN_SEEDS + TOPUP_SEEDS          # n=4
        paths = {s: EUNI_TEST / f'fql_seed{s}.json' for s in seeds}
    else:
        paths = {s: SWEEP_TEST / f'fql_a{atag(alpha)}_seed{s}.json' for s in TRAIN_SEEDS}
    vals = {s: sr(p) for s, p in paths.items()}
    have = [v for v in vals.values() if v is not None]
    return (mean(have) if have else None), len(have), vals

# ReBRAC β1=1.0 clean bar
rb = [sr(Q1C_DIR / f'rebrac_seed{s}.json') for s in TRAIN_SEEDS]
rb = [v for v in rb if v is not None]
BAR = mean(rb) if rb else None
FQL_OLD = 0.855  # FQL α=1.0 clean at n=2 (reference)

all_alphas = sorted(set(ALPHAS_SWEEP) | {ALPHA_BASELINE})
print(f'{"alpha":>7} | {"per-seed":<34} | {"mean":>6} | n')
print('-' * 64)
summary = {}
for a in all_alphas:
    m, n, vals = alpha_cells(a)
    summary[a] = (m, n)
    cells = '  '.join(f'{s}:{("%.2f" % v) if v is not None else "--"}' for s, v in vals.items())
    tag = '  <- frozen (n=4)' if a == ALPHA_BASELINE else ''
    print(f'{a:>7} | {cells:<34} | {("%.3f" % m) if m is not None else "  --  ":>6} | {n}{tag}')

print('-' * 64)
print(f'reference  FQL α=1.0 (old n=2)      = {FQL_OLD:.3f}')
print(f'BAR        ReBRAC β1=1.0 clean      = {BAR:.3f}' if BAR is not None else 'BAR        (missing)')
print('           FQL must reach ~this on clean to have a shot at worst-case > 0.910')

  alpha | per-seed                           |   mean | n
----------------------------------------------------------------
    0.3 | 42:0.75  0:0.73                    |  0.740 | 2
    1.0 | 42:0.80  0:0.91  1:0.91  2:0.81    |  0.858 | 4  <- frozen (n=4)
    3.0 | 42:0.86  0:0.72                    |  0.790 | 2
   10.0 | 42:0.86  0:0.85                    |  0.855 | 2
----------------------------------------------------------------
reference  FQL α=1.0 (old n=2)      = 0.855
BAR        ReBRAC β1=1.0 clean      = 0.910
           FQL must reach ~this on clean to have a shot at worst-case > 0.910


## 7. Verdict (auto) — does any α rescue FQL on the clean axis?

Decision gate(以最佳 α 的 clean mean vs 0.910 bar 为准):
- **RESCUE-PROMISING**: best α clean ≥ **0.90** → 带该 α 进 **Phase C-2**(noisy)确认 worst-case。
- **RESCUE-WEAK**: best α clean ∈ **[0.86, 0.90)** → 有改善但不够;C-2 可选。
- **RESCUE-FAIL**: best α clean < **0.86**(≈ 没超过 α=1.0 自身)→ FQL 过不了 binding axis → **回落 B+A**(诚实机制/负面结果)。

**统计提醒(§9.9)**:sweep α 是 n=2,SE_δ≈4pp;固定 manifest 下对比是配对的,但小差距(<5pp)仍属 NULL 分辨率 — PROMISING 的 α 必须在 C-2 / n=4 复核后才算数。

In [11]:
import json
from pathlib import Path
from statistics import mean

EUNI_TEST  = Path('results/fql_succession/p2/e_uni/test')
Q1C_DIR    = Path('results/fql_succession/p2/e_uni_q1c_actor_pen1_clean/test')
SWEEP_TEST = SWEEP_RES / 'test'

def sr(p): return float(json.loads(p.read_text())['eval_success_rate']) if p.exists() else None
def amean(alpha):
    if alpha == ALPHA_BASELINE:
        paths = [EUNI_TEST / f'fql_seed{s}.json' for s in TRAIN_SEEDS + TOPUP_SEEDS]
    else:
        paths = [SWEEP_TEST / f'fql_a{atag(alpha)}_seed{s}.json' for s in TRAIN_SEEDS]
    vals = [sr(p) for p in paths]
    vals = [v for v in vals if v is not None]
    return mean(vals) if vals else None

rb = [sr(Q1C_DIR / f'rebrac_seed{s}.json') for s in TRAIN_SEEDS]
rb = [v for v in rb if v is not None]
BAR = mean(rb) if rb else 0.910

scored = {a: amean(a) for a in (ALPHAS_SWEEP + [ALPHA_BASELINE])}
have = {a: v for a, v in scored.items() if v is not None}
assert have, 'no FQL clean results found yet — run §4/§5 first'
best_alpha = max(have, key=have.get)
best_clean = have[best_alpha]
a1 = scored.get(ALPHA_BASELINE)

print('FQL clean mean by α:', {a: round(v, 3) for a, v in scored.items() if v is not None})
print(f'best α            = {best_alpha}  (clean μ = {best_clean:.3f})')
print(f'α=1.0 (n=4)       = {a1:.3f}' if a1 is not None else 'α=1.0 (n=4)       = (incomplete)')
print(f'ReBRAC β1=1.0 BAR = {BAR:.3f}')

if best_clean >= 0.90:
    print('\nVERDICT: RESCUE-PROMISING — α={} lifts FQL clean to {:.3f} (≥0.90).'.format(best_alpha, best_clean))
    print('  → Build Phase C-2: run FQL --distill-alpha-bc {} on NOISY M-uni-noise (2 seeds),'.format(best_alpha))
    print('    confirm noisy stays ≥0.91 so worst-case-over-noise > 0.910 (beats ReBRAC β1=1.0).')
    print('    Then take the winning α to n=4 on both axes before any claim.')
elif best_clean >= 0.86:
    print('\nVERDICT: RESCUE-WEAK — best α={} clean {:.3f} in [0.86,0.90).'.format(best_alpha, best_clean))
    print('  Improved over α=1.0 but short of the 0.910 bar. C-2 optional / marginal.')
    print('  Likely still ends at B+A unless C-2 noisy is unexpectedly strong.')
else:
    print('\nVERDICT: RESCUE-FAIL — no α beats FQL α=1.0 clean (~0.855).')
    print('  FQL cannot clear its binding (clean) axis → fall back to B+A:')
    print('  the mechanism trilogy stands, and "FQL got a fair tuned shot and still')
    print('  did not win" is a STRONGER honest-negative than the asymmetric-tuning version.')

FQL clean mean by α: {0.3: 0.74, 3.0: 0.79, 10.0: 0.855, 1.0: 0.858}
best α            = 1.0  (clean μ = 0.858)
α=1.0 (n=4)       = 0.858
ReBRAC β1=1.0 BAR = 0.910

VERDICT: RESCUE-FAIL — no α beats FQL α=1.0 clean (~0.855).
  FQL cannot clear its binding (clean) axis → fall back to B+A:
  the mechanism trilogy stands, and "FQL got a fair tuned shot and still
  did not win" is a STRONGER honest-negative than the asymmetric-tuning version.


## 8. Report checklist

完成后回写主诊断文档:
- [ ] 写入 `docs/fql_succession_p2_mechanism_diagnostic.md` §9.10 — C-1 verdict (per-α clean table + best α + α=1.0 n=4 修正值 + RESCUE 判定)
- [ ] sync `results/fql_succession/p2/e_uni_c1_fql_alpha_sweep/test/*.json`(6 个)+ 新的 `results/fql_succession/p2/e_uni/test/fql_seed{1,2}.json`(2 个)回 local
- [ ] 据 verdict 决定:**PROMISING** → 建 C-2 noisy notebook;**WEAK/FAIL** → 回落 B+A,解冻 N3/N4

Decision impact:
- **RESCUE-PROMISING**: FQL 在公平调参后于 clean 追平 → C-2 验 worst-case;若 worst-case > 0.910,"FQL wins" 主线**复活**(tuned-vs-tuned),N3/N4 按 superiority 框定。
- **RESCUE-WEAK/FAIL**: FQL 公平调参仍不赢 → 锁定 **B+A**。这反而是更强的诚实负面结果(FQL 拿到对称调参机会仍输),且机制三连(§9.4/9.7/9.9)统计显著、可发表。